# Khakas (kjh) — Tokenisation and Translation

Khakas (Siberian branch, Cyrillic script) currently has rule-based tokenisation and prototype-quality morphological analysis. NLLB-200 provides cross-lingual embeddings and machine translation.

In [ ]:
# Install TurkicNLP
# pip install turkicnlp          # core (tokenization, transliteration)
# pip install "turkicnlp[stanza]"  # adds POS, lemma, depparse, NER
# pip install "turkicnlp[nllb]"    # adds cross-lingual embeddings + translation
# pip install "turkicnlp[all]"     # all optional dependencies

In [ ]:
import turkicnlp
from turkicnlp import Pipeline

## 1. Tokenisation

In [ ]:
from turkicnlp.scripts import Script
from turkicnlp.scripts.detector import detect_script
from turkicnlp.scripts.transliterator import Transliterator

# Khakas Cyrillic text
cyrl = "Мин чалтырарға парам."
print("Script Detection:")
print(f"  Detected: {detect_script(cyrl).name}")
print()

# Cyrillic -> Turkic Common Alphabet (Latin)
try:
    t = Transliterator("kjh", source=Script.CYRILLIC, target=Script.COMMON_TURKIC)
    common = t.transliterate(cyrl)
    print(f"Cyrillic:        {cyrl}")
    print(f"Turkic Common:   {common}")
    
    # Reverse: Turkic Common -> Cyrillic
    t_back = Transliterator("kjh", source=Script.COMMON_TURKIC, target=Script.CYRILLIC)
    cyrl_restored = t_back.transliterate(common)
    print(f"Restored:        {cyrl_restored}")
    print(f"Round-trip match: {cyrl == cyrl_restored}")
except Exception as e:
    print(f"⚠ Note: Transliteration to Turkic Common Alphabet (Latin) may not be fully supported for Khakas: {e}")
    print("  For Cyrillic-based languages, use Script.LATIN as alternative")

## 2. Script Detection and Cyrillic ↔ Latin Transliteration

Khakas uses Cyrillic script. The transliteration system enables conversion to Latin for processing.

In [ ]:
turkicnlp.download("kjh")
nlp_tok = Pipeline("kjh", processors=["tokenize"])
doc = nlp_tok("Мин чалтырарға парам.")
print([w.text for w in doc.words])

## 2. Morphological Analysis (Apertium FST — Prototype)

In [ ]:
nlp = Pipeline(
    "kjh",
    processors=["tokenize", "morph"],
    morph_backend="apertium",
)
doc = nlp("Мин чалтырарға парам.")
for w in doc.words:
    print(f"{w.text:<18} lemma={w.lemma} feats={w.feats}")

## 3. Translation via NLLB-200

In [ ]:
turkicnlp.download("kjh", processors=["translate"])
trans = Pipeline("kjh", processors=["translate"], translate_tgt_lang="eng_Latn")
doc = trans("Мин чалтырарға парам.")
print("EN:", doc.translation)